In [ ]:
!pip install langchain langchain-community langchain-text-splitters chromadb sentence-transformers -q

In [ ]:
from langchain_text_splitters import CharacterTextSplitter

with open("/content/drive/MyDrive/rag_notes_qa/data/combined_notes.txt", "r") as f:
    text = f.read()

splitter = CharacterTextSplitter(chunk_size=300, chunk_overlap=50, separator="\n\n")
chunks = splitter.split_text(text)

print(f"Total chunks: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---\n{chunk}")

Total chunks: 9

--- Chunk 1 ---
RAG (Retrieval-Augmented Generation) is a technique that combines information retrieval with text generation. Instead of relying only on what a language model learned during training, RAG retrieves relevant documents from an external knowledge source and uses them as context before generating an answer. This reduces hallucination and allows the model to answer using up-to-date or private data.

--- Chunk 2 ---
The RAG pipeline has three main steps. First, indexing: documents are split into smaller chunks, each chunk is converted into a numeric vector using an embedding model, and these vectors are stored in a vector database. Second, retrieval: when a user asks a question, the question is also converted into an embedding, and the vector database is searched for the most similar chunks using cosine similarity. Third, generation: the retrieved chunks are combined with the original question into a single prompt, which is passed to a language model to produ

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!ls "/content/drive/MyDrive/rag_notes_qa"

combined_notes.txt  rag_notes_qa.ipynb


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vector_db = Chroma.from_texts(chunks, embedding_model)

print("Vector database created successfully")

/tmp/ipykernel_1219/2161308168.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings import HuggingFaceEmbeddings
/tmp/ipykernel_1219/2161308168.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector database created successfully


In [ ]:
query = "What is cosine similarity?"

results = vector_db.similarity_search(query, k=2)

for i, r in enumerate(results):
    print(f"\n--- Retrieved Chunk {i+1} ---\n{r.page_content}")


--- Retrieved Chunk 1 ---
Embeddings are numeric representations of text that capture meaning. Texts with similar meaning have embeddings that are close together in vector space, even if they use completely different words. This is measured using cosine similarity, which calculates the angle between two vectors. A smaller angle means higher similarity.

--- Retrieved Chunk 2 ---
Vector databases such as Chroma, Pinecone, and FAISS are optimized for fast similarity search across large numbers of embeddings, unlike traditional databases which are optimized for exact matches.


In [ ]:
query2 = "How is waste sorted and annotated for AI models?"

results2 = vector_db.similarity_search(query2, k=2)

for i, r in enumerate(results2):
    print(f"\n--- Retrieved Chunk {i+1} ---\n{r.page_content}")


--- Retrieved Chunk 1 ---
High-quality annotated data is essential for waste sorting robots and AI-powered recycling systems to work accurately, reduce contamination rates, and improve overall recycling efficiency.

--- Retrieved Chunk 2 ---
In waste management AI systems, computer vision models are trained to identify and classify different types of waste materials from images captured on conveyor belts, bins, or sorting facilities. Common categories include plastic, paper, metal, glass, organic waste, and e-waste.


In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")

context = "\n".join([r.page_content for r in results2])

prompt = f"""Answer the question using only the context below.

Context:
{context}

Question: {query2}

Answer:"""

inputs = tokenizer(prompt, return_tensors="pt")
outputs = model.generate(**inputs, max_length=100)
answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(answer)

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

images captured on conveyor belts, bins, or sorting facilities
